# CS 418 Project Checkpoint Two: Data Acquisition

Name: Hannan Shahid


## 1. Imports

If this cell runs, Polars is installed in whatever environment this notebook's kernel is using.


In [1]:
import polars as pl

print("polars", pl.__version__)


polars 1.44.2


## 2. Read a dataset

Anything from the `datasets/` folder of the course repo, or any other file you were curious about. The path below is an example; change it to whatever you are actually reading.


In [2]:
df = pl.read_csv("data/results.csv", try_parse_dates=True)
df.shape

(49547, 9)

In [3]:
print(df.schema)
print(df.select(pl.col("date").min().alias("first_match"), pl.col("date").max().alias("last_match")))
print(df["neutral"].value_counts())
print(df.null_count())

Schema({'date': Date, 'home_team': String, 'away_team': String, 'home_score': Int64, 'away_score': Int64, 'tournament': String, 'city': String, 'country': String, 'neutral': Boolean})
shape: (1, 2)
┌─────────────┬────────────┐
│ first_match ┆ last_match │
│ ---         ┆ ---        │
│ date        ┆ date       │
╞═════════════╪════════════╡
│ 1872-11-30  ┆ 2026-08-26 │
└─────────────┴────────────┘
shape: (2, 2)
┌─────────┬───────┐
│ neutral ┆ count │
│ ---     ┆ ---   │
│ bool    ┆ u32   │
╞═════════╪═══════╡
│ false   ┆ 36389 │
│ true    ┆ 13158 │
└─────────┴───────┘
shape: (1, 9)
┌──────┬───────────┬───────────┬────────────┬───┬────────────┬──────┬─────────┬─────────┐
│ date ┆ home_team ┆ away_team ┆ home_score ┆ … ┆ tournament ┆ city ┆ country ┆ neutral │
│ ---  ┆ ---       ┆ ---       ┆ ---        ┆   ┆ ---        ┆ ---  ┆ ---     ┆ ---     │
│ u32  ┆ u32       ┆ u32       ┆ u32        ┆   ┆ u32        ┆ u32  ┆ u32     ┆ u32     │
╞══════╪═══════════╪═══════════╪════════════╪═══╪══

## 3. Describe one or more columns

[DataFrame.describe](https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.describe.html) gives you count, null count, mean, std, and quartiles. Select the column(s) you want first.


In [4]:
df.select("home_score", "away_score").describe()

statistic,home_score,away_score
str,f64,f64
"""count""",49547.0,49547.0
"""null_count""",0.0,0.0
"""mean""",1.75714,1.182715
"""std""",1.773702,1.402353
"""min""",0.0,0.0
"""25%""",1.0,0.0
"""50%""",1.0,1.0
"""75%""",2.0,2.0
"""max""",31.0,21.0


## 4. Filter the dataframe

[DataFrame.filter](https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.filter.html) keeps the rows where an expression is true. Any filter counts — pick one that answers something you were actually wondering about.


In [5]:
home = df.filter(~pl.col("neutral"))
neutral = df.filter(pl.col("neutral"))

for name, d in [("home venue", home), ("neutral venue", neutral)]:
    rate = d.filter(pl.col("home_score") > pl.col("away_score")).height / d.height
    print(f"{name}: {d.height} matches, 'home' team win rate {rate:.1%}")

home venue: 36389 matches, 'home' team win rate 50.7%
neutral venue: 13158 matches, 'home' team win rate 44.2%


## 5. A text cell

Primary dataset 2: International football results (Hannan)
I got this dataset from: (https://github.com/martj42/international_results/blob/master/results.csv). The file is 'results.csv', and it's loaded in 'international_matches.ipynb'.
I picked this dataset because it shows the home, away, and neutral international games, and I found the neutral column to be important because it gives us extra information to look at. It also gives us a no home advantage baseline to compare real home games against, which the Premier League dataset can't do since every league game has a home team.
What's in it:

49,547 rows and 9 columns, and each row is a men’s international match.
It covers 30 Nov 1872 to 26 Aug 2026.
36,389 matches were played at home and 13,158 at a neutral venue.
No missing values in any columns.
Extra time scores are included but not penalty shootouts.
Only the results are recorded, no attendance. 
Columns being used: ‘date’(date), ‘home_team’, ‘away_team’, ‘tournament’, ‘city’,‘country’ (strings) , ‘home_score’ and ‘away_score’(ints), ‘neutral’(boolean).

Quick look: Home teams averaged 1.76 goals vs 1.18 away. At home venues the home team won 50.7%, and at neutral venues the team listed won 44.2%, a gap of about 6.5 points.
